# Agent 12 — Human Review Layer

**What this notebook does:**  
Records every decision where a human team member overrides the quantitative model — whether adding a stock the algorithm ranked low, removing one the algorithm ranked high, or adjusting a weight. Creates an auditable decision log required by the mandate.

**How to present this to investors:**  
> *Our human review layer ensures the pipeline is governed, not just automated. Every deviation from the quantitative model is logged here with a written rationale, the name of the decision-maker, and the date. The panel can inspect any override and challenge the reasoning. This is how we demonstrate that the AI is a tool supporting human judgement, not a black box replacing it.*

**Why this matters for the Q&A:**  
The assignment requires at least 3 concrete human override examples. This notebook is the evidence.  
The panel will specifically ask: *"Give me an example where you overruled the model and why."*

**Run after:** `04_portfolio_construction.ipynb` and `09_greenwashing.ipynb`

## Step 1 — Load the current portfolio and scoring data

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import glob
from datetime import date

TODAY = str(date.today())

# Load the latest portfolio
port_files = sorted(glob.glob("../outputs/portfolio/final_portfolio_*.csv"))
if port_files:
    portfolio = pd.read_csv(port_files[-1])
    print(f"Portfolio loaded: {port_files[-1]}")
    print(f"Holdings: {len(portfolio)}")
    print(portfolio.head())
else:
    portfolio = pd.DataFrame()
    print("No final portfolio found. Run 04_portfolio_construction.ipynb first.")
    print("You can still log overrides now — they will apply when the portfolio is built.")

## Step 2 — Build the override decision log

The log is assembled from two sources:

1. **IC watchlist worksheet** (`ic_overrides_watchlist_*.csv`, produced by NB10 Step 5b) — one row per watchlisted holding that reached the final 20. The mandate requires an Investment Committee override decision, with a documented rationale, before any watchlisted name can be held. These rows load automatically; the **team must complete** `decided_by`, `human_decision`, `rationale` and `approved_by` for each.
2. **Discretionary overrides** — any `ADD` / `REMOVE` / `ADJUST_WEIGHT` / `EXCLUSION_OVERRIDE` decision the team makes that deviates from the model. Fill the `OVERRIDES` list.

The assignment requires **≥ 3 documented override examples**. The watchlist holdings alone exceed that once their rationale is written. An entry counts as *documented* only when both a written rationale and an approver are recorded — the notebook tracks this and flags what is still pending.

> **The rationales must be written by the team, not the AI.** The whole point of this layer — and of the AI Use Statement below — is to evidence human judgement over the pipeline.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  Build the override decision log
# ════════════════════════════════════════════════════════════════════════════

LOG_COLS = ["date", "decided_by", "ticker", "company_name", "override_type",
            "model_decision", "human_decision", "rationale", "evidence", "approved_by"]

# ── Source 1: IC watchlist worksheet from NB10 Step 5b ─────────────────────────
ic_files = sorted(glob.glob("../outputs/scores/ic_overrides_watchlist_*.csv"))
if ic_files:
    ic_log = pd.read_csv(ic_files[-1])
    print(f"IC watchlist worksheet : {ic_files[-1]}")
    print(f"  {len(ic_log)} watchlisted holding(s) requiring an IC override decision")
else:
    ic_log = pd.DataFrame(columns=LOG_COLS)
    print("No IC watchlist worksheet found — run NB10 Step 5b first.")

# ── Source 2: discretionary team overrides ─────────────────────────────────────
# TEAM ACTION: add one dict per deviation from the model. Leave the list empty
# if there are none. Copy this template, uncomment, and fill EVERY field:
#   {
#       "date": TODAY, "decided_by": "<name / role>",
#       "ticker": "<yf_ticker>", "company_name": "<company>",
#       "override_type": "ADD",        # ADD / REMOVE / ADJUST_WEIGHT / EXCLUSION_OVERRIDE
#       "model_decision": "<what the pipeline did, with the number>",
#       "human_decision": "<what the team decided>",
#       "rationale":      "<evidence-based reason>",
#       "evidence":       "<source document, page>",
#       "approved_by":    "<approver>",
#   },
OVERRIDES = [
]
team_log = pd.DataFrame(OVERRIDES, columns=LOG_COLS)

# ── Combine into one log ───────────────────────────────────────────────────────
overrides_df = pd.concat([ic_log.reindex(columns=LOG_COLS), team_log],
                         ignore_index=True)

def _is_filled(v):
    return isinstance(v, str) and v.strip() != ""

# An entry is 'documented' only once a written rationale AND an approver exist
overrides_df["documented"] = overrides_df.apply(
    lambda r: _is_filled(r.get("rationale")) and _is_filled(r.get("approved_by")),
    axis=1)

n_total = len(overrides_df)
n_done  = int(overrides_df["documented"].sum())
print(f"\nOverride log: {n_total} entr{'y' if n_total == 1 else 'ies'} "
      f"— {n_done} documented, {n_total - n_done} pending")
if n_total:
    print("\nBy type:")
    print(overrides_df["override_type"].fillna("(unset)").value_counts().to_string())

## Step 3 — Display the override log

This is the table you reference in Q&A when asked to justify any holding.

In [ ]:
import textwrap

def _show(v):
    return v.strip() if (isinstance(v, str) and v.strip()) else "[TEAM TO COMPLETE]"

print("HUMAN OVERRIDE DECISION LOG")
print("=" * 72)
for i, row in overrides_df.iterrows():
    tag = "DOCUMENTED" if row["documented"] else "PENDING — rationale + sign-off needed"
    print(f"\n  Override #{i+1}   [{tag}]")
    print(f"  {'-'*56}")
    print(f"  Company:       {row['company_name']}  ({row['ticker']})")
    print(f"  Type:          {row['override_type']}")
    print(f"  Date:          {row['date']}")
    print(f"  Decided by:    {_show(row['decided_by'])}")
    print(f"  Approved by:   {_show(row['approved_by'])}")
    print(f"  Model said:    {_show(row['model_decision'])}")
    print(f"  Team decided:  {_show(row['human_decision'])}")
    print(f"  Rationale:")
    for line in textwrap.wrap(_show(row['rationale']), 64):
        print(f"    {line}")
    print(f"  Evidence:      {_show(row['evidence'])}")

print(f"\n{'='*72}")
print(f"Total entries: {n_total}    Documented: {n_done}    Pending: {n_total - n_done}")
print("Mandate requirement: >= 3 documented override examples for the Q&A defence.")
if n_done >= 3:
    print("OK — minimum documented-override requirement met.")
else:
    print(f"ACTION REQUIRED: document at least {3 - n_done} more override(s) "
          f"— fill rationale + approver — before submission.")

## Step 4 — AI Use Statement

Required in the report appendix. This cell generates the standard AI Use Statement for the assignment.

In [ ]:
# Load mandate for fund name
mandate_path = "../outputs/scores/mandate.json"
if os.path.exists(mandate_path):
    with open(mandate_path) as f:
        mandate = json.load(f)
    fund_name = mandate.get("fund_name", "ESADE Sustainable European Equity Fund")
else:
    fund_name = "ESADE Sustainable European Equity Fund"

ai_statement = f"""
AI USE STATEMENT
================

Fund: {fund_name}
Date: {TODAY}

1. TOOLS USED
   - Claude (Anthropic) — mandate drafting, ESG analysis, greenwashing detection
     (8-Test forensic analysis), document intelligence (RAG extraction from PDFs),
     code generation for the pipeline notebooks, and natural language generation
     for report sections
   - Python / Jupyter Notebooks — quantitative data processing, ESG scoring,
     financial metrics calculation, portfolio construction, chart generation
   - yfinance — market price data retrieval
   - n8n.cloud — pipeline orchestration connecting agents

2. WHAT AI DID
   - Generated all Python code in notebooks (team verified outputs, not code)
   - Extracted structured data from PDF sustainability reports with citations
   - Applied the 8-dimension greenwashing test to each company's sustainability claims
   - Drafted and revised report sections based on team-provided data
   - Assisted in interpreting regulatory frameworks (EU Taxonomy, SFDR, CSRD/ESRS)

3. WHAT HUMANS DID
   - Defined all investment mandate parameters and scoring weights
   - Verified 30% random sample of AI-extracted data against source PDFs
   - Verified 100% of exclusion and watchlist decisions against source PDFs
   - Logged {n_total} override / watchlist decision(s), of which {n_done} are
     fully documented with a written rationale and sign-off (see override log above)
   - Reviewed and approved all portfolio construction decisions
   - Validated all financial calculations against source data

4. HALLUCINATION CONTROLS
   - All Claude Projects outputs marked MISSING where information was absent
   - No data was invented, inferred, or synthesised without explicit disclosure
   - Every AI-extracted data point cites verbatim source quote and page number
   - AI-estimated values are classified as 'estimated' in the data dictionary
   - ESG ratings treated as indicators, not objective truth
   - Override rationales are written by the team, not generated by AI

5. LIMITATIONS
   - Biodiversity scores use sector-level proxies (ENCORE / WRI Aqueduct)
     due to absence of company-level TNFD disclosures
   - EU Taxonomy alignment data is sparse; eligibility used as proxy
   - Market data sourced from Yahoo Finance (adjusted prices, potential delays)
   - Some carbon-intensity figures are sector-median imputed where company-level
     data was unavailable (flagged `sector_median_imputed` in the dataset)
   - AI extraction quality depends on PDF text layer quality

This portfolio is an academic prototype produced for ESADE MSc Finance and does
not constitute financial advice or a regulated investment product.
"""

print(ai_statement)

## Step 5 — Save all outputs

In [ ]:
os.makedirs("../outputs/scores", exist_ok=True)
os.makedirs("../outputs/reports", exist_ok=True)

# Save override log (10-column human_overrides schema; drop the helper column)
override_path = f"../outputs/scores/human_overrides_{TODAY}.csv"
overrides_df[LOG_COLS].to_csv(override_path, index=False)
print(f"Override log saved:     {override_path}  ({n_total} entries, {n_done} documented)")

# Save AI Use Statement as text file
statement_path = f"../outputs/reports/ai_use_statement_{TODAY}.txt"
with open(statement_path, "w", encoding="utf-8") as f:
    f.write(ai_statement)
print(f"AI Use Statement saved: {statement_path}")

if n_done < 3:
    print(f"\nREMINDER: {3 - n_done} more override(s) need a rationale + sign-off "
          f"before submission (mandate requires >= 3 documented).")
print("\nHuman review layer complete.")

## ✅ Notebook complete

You now have:
- A **documented override log** with rationale for every deviation from the quantitative model
- An **AI Use Statement** ready to paste into the report appendix
- Evidence that the AI is a tool supporting human judgement, not replacing it

**For Q&A on human oversight:**  
- Open this notebook and point to the override log table
- Be ready to explain Override #1 in detail (pick the most interesting one)
- Emphasise: the model creates a default ranking; humans make the final decision

**Next:** Open `05_reporting.ipynb` to generate charts and the portfolio factsheet.